In [4]:
import cv2
import numpy as np

# ==============================================================================
# 1. Criação do Cenário de Teste (Simulando uma máscara binarizada com ruídos)
# ==============================================================================
# Imagem preta (fundo)
mascara_teste = np.zeros((400, 600), dtype=np.uint8)

# Objeto principal: uma forma sólida branca (simulando um caractere ou peça)
cv2.rectangle(mascara_teste, (150, 100), (320, 280), 255, -1)
cv2.circle(mascara_teste, (320, 190), 50, 255, -1)

# Ruídos espúrios (pequenas manchas que a binarização costuma deixar)
cv2.circle(mascara_teste, (50, 50), 4, 255, -1)
cv2.circle(mascara_teste, (480, 80), 3, 255, -1)
cv2.rectangle(mascara_teste, (400, 320), (410, 328), 255, -1)


# ==============================================================================
# 2. Vetorização: cv2.findContours
# ==============================================================================
# cv2.RETR_EXTERNAL: Recupera apenas as bordas externas mais externas
# cv2.CHAIN_APPROX_SIMPLE: Comprime segmentos redundantes (economiza memória)
contornos, _ = cv2.findContours(
    mascara_teste, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)

# Imagem colorida para desenhar os resultados
canvas_resultado = cv2.cvtColor(mascara_teste, cv2.COLOR_GRAY2BGR)

# ==============================================================================
# 3. Filtragem por Área e Extração de Bounding Box Automática
# ==============================================================================
area_minima = 500  # Limiar para descartar poeira/ruídos pequenos
caixa_detectada = None

for c in contornos:
    area = cv2.contourArea(c)

    if area < area_minima:
        # Pinta ruídos descartados de cinza escuro para visualização
        x_r, y_r, w_r, h_r = cv2.boundingRect(c)
        cv2.rectangle(
            canvas_resultado, (x_r, y_r), (x_r + w_r, y_r + h_r), (80, 80, 80), 1
        )
        continue

    # Objeto válido encontrado: calcula sua Bounding Box
    x, y, w, h = cv2.boundingRect(c)
    caixa_detectada = (x, y, x + w, y + h)

    # Desenha o contorno real em amarelo e a Bounding Box em verde
    cv2.drawContours(canvas_resultado, [c], -1, (0, 255, 255), 2)
    cv2.rectangle(canvas_resultado, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(
        canvas_resultado,
        f"Detectado (Area: {int(area)})",
        (x, y - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 255, 0),
        1,
    )


# ==============================================================================
# 4. Cálculo de IoU (Comparação com Caixa Manual / Ground Truth)
# ==============================================================================
def calcular_iou(box_a, box_b):
    """Calcula a sobreposição entre duas caixas no formato (x1, y1, x2, y2)."""
    # Determina as coordenadas da área de interseção
    x_inter1 = max(box_a[0], box_b[0])
    y_inter1 = max(box_a[1], box_b[1])
    x_inter2 = min(box_a[2], box_b[2])
    y_inter2 = min(box_a[3], box_b[3])

    largura_inter = max(0, x_inter2 - x_inter1)
    altura_inter = max(0, y_inter2 - y_inter1)
    area_intersecao = largura_inter * altura_inter

    # Áreas individuais de cada caixa
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])

    # Área da União = Área(A) + Área(B) - Área(Interseção)
    area_uniao = float(area_a + area_b - area_intersecao)

    if area_uniao == 0:
        return 0.0

    return area_intersecao / area_uniao


# Suponha que um anotador humano marcou manualmente este retângulo original:
caixa_manual = (145, 95, 365, 285)  # (x1, y1, x2, y2)

# Desenha a anotação manual em azul
cv2.rectangle(
    canvas_resultado,
    (caixa_manual[0], caixa_manual[1]),
    (caixa_manual[2], caixa_manual[3]),
    (255, 100, 0),
    2,
)
cv2.putText(
    canvas_resultado,
    "Ground Truth (Manual)",
    (caixa_manual[0], caixa_manual[3] + 20),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.5,
    (255, 100, 0),
    1,
)

# Calcula o IoU se o algoritmo detectou o objeto
if caixa_detectada is not None:
    valor_iou = calcular_iou(caixa_detectada, caixa_manual)
    print(f"Resultado do IoU entre Algoritmo e Anotação Manual: {valor_iou:.4f}")

    # Exibe a pontuação de IoU na própria imagem
    cv2.putText(
        canvas_resultado,
        f"IoU: {valor_iou * 100:.1f}%",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0, 255, 255),
        2,
    )

# Exibição
cv2.imshow("Mascara Binaria de Entrada", mascara_teste)
cv2.imshow("Vetorizacao, Bounding Box e IoU", canvas_resultado)
cv2.waitKey(0)
cv2.destroyAllWindows()

Resultado do IoU entre Algoritmo e Anotação Manual: 0.9074
